# QA 数据查看器

用于查看和检查 QA 数据集，包括图片、问题、答案和模型回答


In [ ]:
import json
import random
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
import warnings
warnings.filterwarnings('ignore')


In [ ]:
json_file_path = "/home/zenglingfeng/qa_pipline12-7/evaluate/outputs/expert/Llama-3.2-11B-Vision/last_evaluate.jsonl"
# 也支持 JSONL 格式：
#json_file_path = "/home/zenglingfeng/qa_pipline/output/测试.jsonl"

# 只存储文件路径和总数量，不一次性加载所有数据（节省内存）
def get_data_count(file_path):
    """获取数据总数，不加载实际数据"""
    if file_path.lower().endswith('.jsonl'):
        count = 0
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('{"statistics"'):
                    try:
                        json.loads(line)
                        count += 1
                    except:
                        pass
        return count
    else:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, dict) and "items" in data:
            return len(data["items"])
        elif isinstance(data, list):
            return len(data)
        return 0

def load_single_item(file_path, index):
    """按需加载单个题目（节省内存）"""
    if file_path.lower().endswith('.jsonl'):
        current_index = 0
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('{"statistics"'):
                    try:
                        if current_index == index:
                            return json.loads(line)
                        current_index += 1
                    except:
                        pass
        return None
    else:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, dict) and "items" in data:
            items = data["items"]
        elif isinstance(data, list):
            items = data
        else:
            return None
        if 0 <= index < len(items):
            return items[index]
        return None

# 获取总数
total_count = get_data_count(json_file_path)
print(f"📊 数据文件包含 {total_count} 个问题（按需加载，节省内存）")


不需要的字段可以在下面代码里注释

In [ ]:
def display_question(item, show_image=True, show_models=True, current_index=0, total_count=0):
    """
    显示单个问题的详细信息（左图右文布局）
    
    参数:
        item: JSON 中的一个问题项
        show_image: 是否显示图片
        show_models: 是否显示模型回答
    """
    from IPython.display import display, HTML
    import base64
    import os
    
    # 收集所有文字信息
    text_lines = []
    
    # 基本信息
    text_lines.append("="*80)
    text_lines.append(f"📝 ID: {item['question_id']}")
    text_lines.append(f"📊 类型: {item['question_type']}")
    # 可选字段：如果存在则显示
    if 'image_type' in item:
        text_lines.append(f"🖼️ 图片类型: {item['image_type']}")
    if 'profile' in item:
        text_lines.append(f"👤 用户画像: {item['profile']}")
    if 'difficulty' in item:
        text_lines.append(f"📈 难度: {item['difficulty']}")
    if 'language' in item:
        text_lines.append(f"🌐 语言: {item['language']}")
    text_lines.append("="*80)
    text_lines.append("")
    
    # 问题
    text_lines.append("【问题】")
    question = item['question']
    if isinstance(question, dict):
        for round_key, round_question in question.items():
            text_lines.append(f"{round_key}:")
            text_lines.append(f"  {round_question}")
            text_lines.append("")
    else:
        text_lines.append(f"  {question}")
        text_lines.append("")
    
    # 选项
    if item.get('options'):
        text_lines.append("【选项】")
        options = item['options']
        if isinstance(options, dict):
            for key, value in options.items():
                text_lines.append(f"  {key}: {value}")
        else:
            text_lines.append(f"  {options}")
        text_lines.append("")
    
    # 标准答案
    text_lines.append("【标准答案】")
    answer = item['answer']
    if isinstance(answer, dict):
        for round_key, round_answer in answer.items():
            text_lines.append(f"  {round_key}: {round_answer}")
    else:
        text_lines.append(f"  {answer}")
    text_lines.append("")
    
    # 出题过程
    if 'qa_make_process' in item:
        text_lines.append("【出题过程】")
        qa_process = item['qa_make_process']
        if isinstance(qa_process, dict):
            for round_key, process in qa_process.items():
                text_lines.append(f"{round_key}:")
                text_lines.append(f"  {process}")
                text_lines.append("")
        else:
            text_lines.append(f"  {qa_process}")
        text_lines.append("")
    
    # 模型回答
    if show_models:
        for model_key in ['model1', 'model2', 'model3']:
            if model_key in item and item[model_key]:
                model_data = item[model_key]
                # 检查是否是有效的模型数据（包含必要字段）
                if isinstance(model_data, dict) and 'model_name' in model_data:
                    match_status = "✅正确" if model_data.get('match_gt', False) else "❌错误"
                    response_time = model_data.get('response_time', 0.0)
                    text_lines.append("─"*80)
                    text_lines.append(f"🤖 {model_data.get('model_name', model_key).upper()} {match_status} | 响应: {response_time:.2f}秒")
                    text_lines.append("")
                    '''
                    # 模型回答输出（process）
                    if 'process' in model_data:
                        text_lines.append("【回答过程】")
                        process = model_data['process']
                        if isinstance(process, dict):
                            for round_key, round_process in process.items():
                                text_lines.append(f"{round_key}:")
                                text_lines.append(f"  {round_process}")
                                text_lines.append("")
                        else:
                            text_lines.append(f"  {process}")
                            text_lines.append("")
                    '''
                    # 模型答案
                    text_lines.append("【最终答案】")
                    model_answer = model_data.get('answer', '')
                    if isinstance(model_answer, dict):
                        for round_key, round_answer in model_answer.items():
                            text_lines.append(f"  {round_key}: {round_answer}")
                    else:
                        text_lines.append(f"  {model_answer}")
                    text_lines.append("")
                    
                    # 判断理由
                    if 'judge_reasoning' in model_data and model_data['judge_reasoning']:
                        text_lines.append(f"【判断理由】{model_data['judge_reasoning']}")
                        text_lines.append("")
    
    # 分类信息
    if 'classification' in item:
        classification = item['classification']
        text_lines.append("─"*80)
        text_lines.append(f"📊 级别: {classification['level']} | 类别: {classification['category']} | 一致数: {classification['agreement_count']}")
    
    text_lines.append("="*80)
    
    # 处理图片路径：支持多张图片
    image_paths = []
    image_path_raw = item.get('image_path', '')
    if isinstance(image_path_raw, list):
        image_paths = [path for path in image_path_raw if path]
    elif isinstance(image_path_raw, str) and image_path_raw:
        image_paths = [image_path_raw]
    
    # 构建图片HTML（支持多张图片横向滚动）
    images_html = ""
    if show_image and image_paths:
        images_list = []
        for idx, image_path in enumerate(image_paths):
            try:
                # 使用PIL检测真实的图片格式
                from PIL import Image
                with Image.open(image_path) as img:
                    img_format = img.format.lower() if img.format else 'png'
                    if img_format == 'jpg':
                        img_format = 'jpeg'
                    elif img_format not in ['png', 'jpeg', 'gif', 'webp']:
                        img_format = 'png'
                
                # 将图片转换为base64
                with open(image_path, 'rb') as f:
                    img_data = f.read()
                    img_base64 = base64.b64encode(img_data).decode()
                
                img_ext = img_format
                if not img_ext:
                    img_ext = image_path.split('.')[-1].lower()
                    if img_ext == 'jpg':
                        img_ext = 'jpeg'
                    if img_ext not in ['png', 'jpeg', 'gif', 'webp']:
                        img_ext = 'png'
                
                image_filename = os.path.basename(image_path)
                image_label = f"图片 {idx+1}/{len(image_paths)}" if len(image_paths) > 1 else "图片"
                
                images_list.append(f'''
                    <div style="flex: 0 0 auto; margin-right: 20px; text-align: center;">
                        <div style="font-weight: bold; margin-bottom: 8px; color: #666;">{image_label}</div>
                        <img src="data:image/{img_ext};base64,{img_base64}" 
                             style="max-width: 100%; max-height: 600px; width: auto; height: auto; 
                                    border: 1px solid #ddd; border-radius: 4px; display: block;
                                    image-rendering: auto; object-fit: contain;">
                        <div style="margin-top: 10px;">
                            <a href="data:image/{img_ext};base64,{img_base64}" 
                               download="{image_filename}"
                               style="display: inline-block; padding: 6px 12px; background-color: #4CAF50; 
                                      color: white; text-decoration: none; border-radius: 4px; font-size: 12px;
                                      cursor: pointer; margin-right: 8px;"
                               onmouseover="this.style.backgroundColor='#45a049'"
                               onmouseout="this.style.backgroundColor='#4CAF50'">
                                📥 下载
                            </a>
                            <button onclick="
                                (async function() {{
                                    try {{
                                        const btn = event.target;
                                        const img = btn.parentElement.previousElementSibling;
                                        const imgSrc = img.src;
                                        const tempImg = new Image();
                                        tempImg.crossOrigin = 'anonymous';
                                        await new Promise((resolve, reject) => {{
                                            tempImg.onload = resolve;
                                            tempImg.onerror = reject;
                                            tempImg.src = imgSrc;
                                        }});
                                        const canvas = document.createElement('canvas');
                                        canvas.width = tempImg.naturalWidth;
                                        canvas.height = tempImg.naturalHeight;
                                        const ctx = canvas.getContext('2d');
                                        ctx.drawImage(tempImg, 0, 0);
                                        canvas.toBlob(async (blob) => {{
                                            try {{
                                                await navigator.clipboard.write([
                                                    new ClipboardItem({{'image/png': blob}})
                                                ]);
                                                const originalText = btn.innerHTML;
                                                btn.innerHTML = '✅ 已复制!';
                                                btn.style.backgroundColor = '#45a049';
                                                setTimeout(() => {{
                                                    btn.innerHTML = originalText;
                                                    btn.style.backgroundColor = '#2196F3';
                                                }}, 2000);
                                            }} catch (err) {{
                                                alert('复制失败: ' + err.message);
                                            }}
                                        }}, 'image/png');
                                    }} catch (err) {{
                                        alert('复制失败: ' + err.message);
                                    }}
                                }})();
                            " 
                            style="display: inline-block; padding: 6px 12px; background-color: #2196F3; 
                                   color: white; border: none; border-radius: 4px; font-size: 12px;
                                   cursor: pointer;"
                            onmouseover="this.style.backgroundColor='#1976D2'"
                            onmouseout="this.style.backgroundColor='#2196F3'">
                                📋 复制
                            </button>
                        </div>
                    </div>
                ''')
            except Exception as e:
                images_list.append(f'<div style="color: red;">❌ 无法加载图片 {idx+1}: {image_path} | 错误: {e}</div>')
        
        if images_list:
            images_html = f'''
            <div style="overflow-x: auto; margin-bottom: 20px; padding: 10px; background: #f5f5f5; border-radius: 4px;">
                <div style="display: flex; gap: 20px;">
                    {''.join(images_list)}
                </div>
            </div>
            '''
    
    # 构建进度信息（导航按钮由ipywidgets提供）
    progress_html = ""
    if total_count > 0:
        progress = (current_index + 1) / total_count * 100
        progress_html = f'''
        <div style="background: #f0f0f0; padding: 10px; margin-bottom: 15px; border-radius: 4px; border: 1px solid #ddd;">
            <div style="font-weight: bold; margin-bottom: 8px; font-size: 14px;">
                题目 {current_index + 1} / {total_count}
            </div>
            <div style="background: #ddd; height: 20px; border-radius: 10px; overflow: hidden;">
                <div style="background: #4CAF50; height: 100%; width: {progress}%; 
                            transition: width 0.3s; display: flex; align-items: center; 
                            justify-content: center; color: white; font-size: 11px;">
                    {progress:.1f}%
                </div>
            </div>
        </div>
        '''
    
    # 构建完整HTML
    text_content = '\n'.join(text_lines)
    html = f'''
    {progress_html}
    <div style="display: flex; gap: 20px; align-items: flex-start; width: 100%;">
        <div style="flex: 1; max-width: {'60%' if show_image and image_paths else '100%'};">
            {images_html if show_image and image_paths else ''}
        </div>
        <div style="flex: 1; max-width: {'40%' if show_image and image_paths else '100%'}; 
                    overflow-y: auto; max-height: 800px;">
            <pre style="font-size: 12px; line-height: 1.4; white-space: pre-wrap; 
                        word-wrap: break-word; font-family: monospace; margin: 0;">{text_content}</pre>
        </div>
    </div>
    '''
    
    if not show_image or not image_paths:
        html = f'''
        {progress_html}
        <pre style="font-size: 12px; line-height: 1.4; white-space: pre-wrap; 
                    word-wrap: break-word; font-family: monospace; margin: 0;">{text_content}</pre>
        '''
    
    display(HTML(html))


### 查看模式选择

支持多种查看模式：
- **`interactive`**: 交互式查看器（一次只显示一道题，节省内存）
  - ✅ 多张图片横向滚动显示
  - ✅ 上一题/下一题导航按钮
  - ✅ 进度条显示当前位置
  - ✅ 按需加载，节省内存
- **`single`**: 查看指定索引的问题
- **`sequential`**: 顺序查看前N个问题
- **`random`**: 随机查看N个问题
- **`filter`**: 筛选特定类型的问题


In [ ]:
# ============================================================================
# 配置参数
# ============================================================================
MODE = 'interactive'  # 可选: 'interactive', 'single', 'sequential', 'random', 'filter'
SHOW_IMAGE = True     # 是否显示图片
SHOW_MODELS = True    # 是否显示模型回答

# 以下参数根据MODE使用：
INDEX = 0             # 当 MODE='single' 时使用
COUNT = 10            # 当 MODE='sequential' 或 'random' 时使用
QUESTION_TYPE_FILTER = "问答题"  # 当 MODE='filter' 时使用
START_INDEX = 0       # 当 MODE='interactive' 时使用，起始题目索引

# ============================================================================
# 交互式查看器（MODE='interactive'）
# ============================================================================
if MODE == 'interactive':
    # 全局变量：当前题目索引
    current_index = START_INDEX

    def show_question(index):
        """显示指定索引的题目（交互式模式）"""
        global current_index
        if index < 0 or index >= total_count:
            print(f"❌ 索引 {index} 超出范围（总共 {total_count} 个问题）")
            return
        
        current_index = index
        clear_output(wait=True)
        
        # 按需加载当前题目
        item = load_single_item(json_file_path, index)
        if item:
            display_question(item, show_image=SHOW_IMAGE, show_models=SHOW_MODELS, 
                            current_index=index, total_count=total_count)
        else:
            print(f"❌ 无法加载题目 {index}")

    def create_viewer():
        """创建交互式查看器"""
        global current_index
        
        # 创建控件
        index_slider = widgets.IntSlider(
            value=START_INDEX,
            min=0,
            max=max(0, total_count - 1),
            step=1,
            description='题目索引:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='500px')
        )
        
        prev_button = widgets.Button(
            description='⬅️ 上一题',
            button_style='info',
            layout=widgets.Layout(width='120px')
        )
        
        next_button = widgets.Button(
            description='下一题 ➡️',
            button_style='info',
            layout=widgets.Layout(width='120px')
        )
        
        jump_button = widgets.Button(
            description='跳转',
            button_style='success',
            layout=widgets.Layout(width='80px')
        )
        
        index_input = widgets.IntText(
            value=START_INDEX + 1,
            description='跳转到:',
            style={'description_width': 'initial'},
            layout=widgets.Layout(width='150px')
        )
        
        # 输出区域
        output = widgets.Output()
        
        # 更新标签的函数
        def update_label():
            label.value = f'题目 {current_index + 1} / {total_count}'
        
        label = widgets.Label(
            f'题目 {current_index + 1} / {total_count}', 
            layout=widgets.Layout(width='150px', margin='0 10px')
        )
        
        # 事件处理函数
        def on_slider_change(change):
            global current_index
            current_index = change['new']
            index_input.value = current_index + 1
            update_label()
            show_question(change['new'])
        
        def on_prev_click(b):
            global current_index
            if current_index > 0:
                new_index = current_index - 1
                current_index = new_index
                index_slider.value = new_index
                index_input.value = new_index + 1
                update_label()
                show_question(new_index)
        
        def on_next_click(b):
            global current_index
            if current_index < total_count - 1:
                new_index = current_index + 1
                current_index = new_index
                index_slider.value = new_index
                index_input.value = new_index + 1
                update_label()
                show_question(new_index)
        
        def on_jump_click(b):
            global current_index
            target = index_input.value - 1
            if 0 <= target < total_count:
                current_index = target
                index_slider.value = target
                update_label()
                show_question(target)
            else:
                with output:
                    print(f"❌ 索引 {target} 超出范围（总共 {total_count} 个问题）")
        
        # 绑定事件
        index_slider.observe(on_slider_change, names='value')
        prev_button.on_click(on_prev_click)
        next_button.on_click(on_next_click)
        jump_button.on_click(on_jump_click)
        
        # 布局
        controls = widgets.HBox([
            prev_button,
            label,
            next_button,
            widgets.Label('|', layout=widgets.Layout(margin='0 10px')),
            index_input,
            jump_button,
            widgets.Label('|', layout=widgets.Layout(margin='0 10px')),
            index_slider
        ])
        
        # 显示初始题目
        with output:
            show_question(START_INDEX)
        
        # 返回控件和输出
        display(controls, output)

    # 启动交互式查看器
    create_viewer()

# ============================================================================
# 其他查看模式（批量显示）
# ============================================================================
else:
    # 加载所有数据（用于批量显示模式）
    def load_all_data(file_path):
        """加载所有数据（用于批量显示模式）"""
        if file_path.lower().endswith('.jsonl'):
            data = []
            with open(file_path, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith('{"statistics"'):
                        try:
                            item = json.loads(line)
                            data.append(item)
                        except json.JSONDecodeError as e:
                            print(f"⚠️ 警告：跳过无效行: {e}")
            return data
        else:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            if isinstance(data, dict) and "items" in data:
                return data["items"]
            elif isinstance(data, list):
                return data
            else:
                raise ValueError("输入 JSON 格式不正确")
    
    data = load_all_data(json_file_path)
    print(f"📊 已加载 {len(data)} 个问题（批量显示模式）\n")
    
    if MODE == 'single':
        # 1. 查看指定索引的问题
        if 0 <= INDEX < len(data):
            print(f"查看第 {INDEX+1} 个问题（索引: {INDEX}）\n")
            display_question(data[INDEX], show_image=SHOW_IMAGE, show_models=SHOW_MODELS, 
                           current_index=INDEX, total_count=len(data))
        else:
            print(f"❌ 索引 {INDEX} 超出范围（总共 {len(data)} 个问题）")

    elif MODE == 'sequential':
        # 2. 顺序查看前N个问题
        n = min(COUNT, len(data))
        for i in range(n):
            print(f"\n\n{'#'*80}")
            print(f"# 第 {i+1} 个问题（索引: {i}）")
            print(f"{'#'*80}\n")
            display_question(data[i], show_image=SHOW_IMAGE, show_models=SHOW_MODELS, 
                           current_index=i, total_count=len(data))

    elif MODE == 'random':
        # 3. 随机查看N个问题
        n = min(COUNT, len(data))
        random_indices = random.sample(range(len(data)), n)
        print(f"随机选择的索引: {random_indices}\n")
        
        for idx, i in enumerate(random_indices, 1):
            print(f"\n\n{'#'*80}")
            print(f"# 第 {idx} 个随机问题（索引: {i}）")
            print(f"{'#'*80}\n")
            display_question(data[i], show_image=SHOW_IMAGE, show_models=SHOW_MODELS, 
                           current_index=i, total_count=len(data))

    elif MODE == 'filter':
        # 4. 筛选特定类型的问题
        filtered_data = [item for item in data if item.get('question_type') == QUESTION_TYPE_FILTER]
        print(f"找到 {len(filtered_data)} 个 '{QUESTION_TYPE_FILTER}' 类型的问题\n")
        
        if filtered_data:
            n = min(COUNT, len(filtered_data))
            for idx, item in enumerate(filtered_data[:n], 1):
                original_index = data.index(item)
                print(f"\n\n{'#'*80}")
                print(f"# 第 {idx} 个 '{QUESTION_TYPE_FILTER}'（原始索引: {original_index}）")
                print(f"{'#'*80}\n")
                display_question(item, show_image=SHOW_IMAGE, show_models=SHOW_MODELS, 
                               current_index=original_index, total_count=len(data))
        else:
            print(f"❌ 没有找到 '{QUESTION_TYPE_FILTER}' 类型的问题")

    else:
        print(f"❌ 未知的查看模式: {MODE}")
        print("支持的模式: 'interactive', 'single', 'sequential', 'random', 'filter'")


In [ ]:
# 统计问题类型分布（按需加载，节省内存）
from collections import Counter

def get_statistics(file_path):
    """统计问题类型分布（不加载所有数据到内存）"""
    question_types = []
    image_types = []
    levels = []
    
    if file_path.lower().endswith('.jsonl'):
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith('{"statistics"'):
                    try:
                        item = json.loads(line)
                        if 'question_type' in item:
                            question_types.append(item['question_type'])
                        if 'image_type' in item:
                            image_types.append(item['image_type'])
                        if 'classification' in item and 'level' in item['classification']:
                            levels.append(item['classification']['level'])
                    except:
                        pass
    else:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, dict) and "items" in data:
            items = data["items"]
        elif isinstance(data, list):
            items = data
        else:
            items = []
        
        for item in items:
            if 'question_type' in item:
                question_types.append(item['question_type'])
            if 'image_type' in item:
                image_types.append(item['image_type'])
            if 'classification' in item and 'level' in item['classification']:
                levels.append(item['classification']['level'])
    
    return question_types, image_types, levels

question_types, image_types, levels = get_statistics(json_file_path)

print("问题类型分布:")
type_counts = Counter(question_types)
for qtype, count in type_counts.items():
    print(f"  {qtype}: {count} 个")

print("\n图片类型分布:")
image_type_counts = Counter(image_types)
for itype, count in image_type_counts.items():
    print(f"  {itype}: {count} 个")

if levels:
    print("\n分类级别分布:")
    level_counts = Counter(levels)
    for level, count in level_counts.items():
        print(f"  {level}: {count} 个")
